# Day 57 — Security, data privacy & ethical considerations
Objectives:
- Identify PII and sensitive attributes.
- Basic de-identification and minimization strategies.
- Ethical considerations, fairness checks, and documentation.
Note: This notebook demonstrates lightweight checks; real programs require legal/policy review.

In [ ]:
import re, pandas as pd
from pathlib import Path
sample = pd.DataFrame({
    'name': ['Alice Smith','Bob Jones'],
    'email': ['alice@example.com','bob@company.org'],
    'phone': ['+1-415-555-1212','(212) 555-9898'],
    'notes': ['Met on 2025-01-01','Lives near 5th Ave']
})
sample


## PII detection (simple regex demo)
Caution: regex is imperfect; use specialized tools for robust detection.

In [ ]:
EMAIL_RE = re.compile(r'[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}')
PHONE_RE = re.compile(r'(?:\+?\d{1,3}[-.\s]?)?(?:\(\d{3}\)|\d{3})[-.\s]?\d{3}[-.\s]?\d{4}')
def find_pii(s: str) -> dict:
    emails = EMAIL_RE.findall(s)
    phones = PHONE_RE.findall(s)
    return {'emails': emails, 'phones': phones}

sample['pii'] = sample.apply(lambda r: {
    'email': EMAIL_RE.findall(r['email']),
    'phone': PHONE_RE.findall(r['phone']),
    'notes': find_pii(r['notes'])
}, axis=1)
sample[['pii']]


## De-identification strategies
- Remove direct identifiers (name, email, phone).
- Pseudonymize with stable hashes.
- Mask partial values.
- Limit retention and access (data minimization).
Below: pseudonymize names with a salted hash.

In [ ]:
import hashlib
SALT = b'secret-salt'  # store securely via environment vars/secret manager
def pseudo(value: str) -> str:
    h = hashlib.sha256(SALT + value.encode()).hexdigest()[:10]
    return f'id_{h}'

redacted = sample.copy()
redacted['name_pseudo'] = redacted['name'].map(pseudo)
redacted = redacted.drop(columns=['name','email','phone'])
redacted


## Fairness and bias (brief)
- Check subgroup performance metrics (e.g., by sex, race, age bucket).
- Avoid using protected attributes directly unless justified; document rationale.
- Consider disparate impact, calibration across groups.
- Provide model cards and data statements.

## Exercises
1) Build a PII scanning utility for a DataFrame (columns + free text).
2) Add a redaction function that masks emails/phones in free text.
3) Simulate subgroup metrics (e.g., by sex) for a classifier and compare precision/recall across groups.
4) Draft a one-page data ethics checklist for your project.